In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC

In [2]:
# 1. Load the datasets
# Make sure train.csv and test.csv are in the same folder as this script
try:
    train_df = pd.read_csv('train.csv')
    test_df = pd.read_csv('test.csv')
    print("Files loaded successfully.")
except FileNotFoundError:
    print("Error: train.csv or test.csv not found in the current directory.")
    exit()

Files loaded successfully.


In [3]:
# 2. Feature Engineering Function
def feature_engineering(df):
    """
    Transforms raw data based on the dataset descriptions:
    - Splits Cabin (deck/num/side)
    - Extracts Group from PassengerId
    - Calculates Total Spending
    """
    # Create a copy to avoid SettingWithCopyWarning
    df = df.copy()
    
    # Split Cabin into three separate features
    # Format: deck/num/side (e.g., B/0/P)
    df[['Deck', 'Num', 'Side']] = df['Cabin'].str.split('/', expand=True)
    
    # Extract Group ID (gggg) from PassengerId (gggg_pp)
    df['Group'] = df['PassengerId'].apply(lambda x: x.split('_')[0])
    
    # Sum luxury amenities to create a 'TotalSpent' feature
    spending_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
    df[spending_cols] = df[spending_cols].fillna(0) # Fill missing spend with 0 before summing
    df['TotalSpent'] = df[spending_cols].sum(axis=1)
    
    # Drop columns that are unique identifiers or have too many categories for a basic SVM
    return df.drop(['PassengerId', 'Cabin', 'Name', 'Num'], axis=1)

# Apply feature engineering
train_processed = feature_engineering(train_df)
test_passenger_ids = test_df['PassengerId'] # Keep for final submission
test_processed = feature_engineering(test_df)

# Separate features (X) and target (y)
X = train_processed.drop('Transported', axis=1)
y = train_processed['Transported'].astype(int) # Convert True/False to 1/0 for SVM

In [4]:
# 3. Preprocessing Pipeline
# SVM is distance-based, so Scaling and One-Hot Encoding are mandatory.
numeric_features = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'TotalSpent']
categorical_features = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'Deck', 'Side']

# Numerical: Fill missing with median and Scale (crucial for SVM)
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical: Fill missing with most frequent and convert to numbers
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combine transformers
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

In [5]:
# 4. Model Definition & GridSearchCV
# Using probability=True allows for better evaluation metrics later if needed.
svm_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', SVC(probability=True))
])

# Hyperparameter Tuning Grid
# We test 3 values for C and 3 for gamma, totaling 9 combinations.
# With 5-fold cross-validation, this will perform 45 total fits.
param_grid = {
    'classifier__C': [0.1, 1, 10],
    'classifier__gamma': ['scale', 0.01, 0.1],
    'classifier__kernel': ['rbf'] 
}

# n_jobs=-1 uses all CPU cores. 
# verbose=2 ensures you see the progress of every single 'fit' in your terminal.
print("\n--- Starting Grid Search ---")
print("Total candidates: 9, Total fits: 45")
grid_search = GridSearchCV(
    svm_pipeline, 
    param_grid, 
    cv=5, 
    scoring='accuracy', 
    n_jobs=-1, 
    verbose=2 
)

# This is where the heavy calculation happens
grid_search.fit(X, y)


--- Starting Grid Search ---
Total candidates: 9, Total fits: 45
Fitting 5 folds for each of 9 candidates, totalling 45 fits
[CV] END classifier__C=0.1, classifier__gamma=scale, classifier__kernel=rbf; total time=  17.9s
[CV] END classifier__C=0.1, classifier__gamma=scale, classifier__kernel=rbf; total time=  18.4s
[CV] END classifier__C=0.1, classifier__gamma=scale, classifier__kernel=rbf; total time=  18.9s
[CV] END classifier__C=0.1, classifier__gamma=scale, classifier__kernel=rbf; total time=  19.0s
[CV] END classifier__C=0.1, classifier__gamma=scale, classifier__kernel=rbf; total time=  19.2s
[CV] END classifier__C=0.1, classifier__gamma=0.01, classifier__kernel=rbf; total time=  21.7s
[CV] END classifier__C=0.1, classifier__gamma=0.01, classifier__kernel=rbf; total time=  21.7s
[CV] END classifier__C=0.1, classifier__gamma=0.01, classifier__kernel=rbf; total time=  21.8s
[CV] END classifier__C=0.1, classifier__gamma=0.1, classifier__kernel=rbf; total time=  17.1s
[CV] END classi

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='median')),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         ['Age',
                                                                          'RoomService',
                                                                          'FoodCourt',
                                                                          'ShoppingMall',
                                                                          'Spa',
                                                                          'VRDeck',
                                                                          'TotalSpent']),
                                                                        ('cat',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='most_frequent')),
                                                                                         ('onehot',
                                                                                          OneHotEncoder(handle_unknown='ignore'))]),
                                                                         ['HomePlanet',
                                                                          'CryoSleep',
                                                                          'Destination',
                                                                          'VIP',
                                                                          'Deck',
                                                                          'Side'])])),
                                       ('classifier', SVC(probability=True))]),
             n_jobs=-1,
             param_grid={'classifier__C': [0.1, 1, 10],
                         'classifier__gamma': ['scale', 0.01, 0.1],
                         'classifier__kernel': ['rbf']},
             scoring='accuracy', verbose=2)

In [6]:
# 5. Results and Evaluation
print("\n--- Grid Search Complete ---")
print(f"Best Parameters Found: {grid_search.best_params_}")
print(f"Best Cross-Validation Accuracy: {grid_search.best_score_:.4f}")


--- Grid Search Complete ---
Best Parameters Found: {'classifier__C': 1, 'classifier__gamma': 0.1, 'classifier__kernel': 'rbf'}
Best Cross-Validation Accuracy: 0.7978


In [7]:
# 6. Generate Kaggle Submission File
print("\nPredicting on test set and generating submission file...")
final_predictions = grid_search.predict(test_processed)
final_predictions_bool = final_predictions.astype(bool) # Convert 1/0 back to True/False

# Create the final DataFrame
submission = pd.DataFrame({
    "PassengerId": test_passenger_ids,
    "Transported": final_predictions_bool
})

# Save to CSV
output_filename = 'submission_svm.csv'
submission.to_csv(output_filename, index=False)
print(f"Success! {output_filename} has been created.")


Predicting on test set and generating submission file...
Success! submission_svm.csv has been created.
